# PEST First Slice Workflow

This notebook demonstrates the first reusable `myflopy` PEST/pyEMU slice:

- GIS-defined `K` zones
- GIS-defined drains
- `HeadTargets` for calibration targets
- `PestProject` template generation
- forward-run application of `K` pilot-point multipliers and `DRN` elevation/conductance parameters

The example is intentionally small, but it follows the same GIS-first pattern we want for larger Cumberland-style models.


In [ ]:
from __future__ import annotations

import shutil
import subprocess
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import Point, Polygon

project_root = Path.cwd().resolve().parents[2]
src = project_root / 'src'
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

import myflopy as mf
from myflopy.modflow.mf6.simulation.packages import CHD, Drains, InitialConditions, KFlow, OutputControl, Storage


In [ ]:
def write_gpkg(path: Path, gdf: gpd.GeoDataFrame) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    gdf.to_file(path, driver='GPKG')
    return path


def two_cell_vor_clockwise():
    verts = np.array(
        [
            [0.0, 0.0],
            [1.0, 0.0],
            [1.0, 1.0],
            [0.0, 1.0],
            [2.0, 0.0],
            [2.0, 1.0],
        ],
        dtype=float,
    )
    iverts = [[0, 3, 2, 1], [1, 2, 5, 4]]
    xcyc = np.array([[0.5, 0.5], [1.5, 0.5]], dtype=float)
    return mf.VoronoiGridPlus(verts=verts, iverts=iverts, xcyc=xcyc)


def build_two_cell_pest_forward_model(name: str, workspace: Path):
    vor = two_cell_vor_clockwise()
    vor.gdf_topbtm = gpd.GeoDataFrame(
        {
            'geometry': vor.gdf_vorPolys.geometry,
            0: [10.0, 10.0],
            1: [0.0, 0.0],
        },
        geometry='geometry',
        crs=vor.crs,
    )
    model = mf.SimulationBase(name=name, mf_folder_path=workspace, vor=vor, nper=1)
    mf.modflow.mf6.DisvGrid(vor=vor, model=model, top=[10.0, 10.0], bottom=[[0.0, 0.0]], nlay=1, idomain=[[1, 1]])
    mf.modflow.mf6.TemporalDiscretization(model=model, per_len=1, num_steps=1, multiplier=1.0)
    InitialConditions(model=model, vor=vor, nlay=1, strt=[10.0, 9.0])
    KFlow(model=model, k=[5.0, 5.0], k33_vert=[1.0, 1.0], save_specific_discharge=False)
    Storage(model=model, sto_steady={0: True}, sto_transient={})
    OutputControl(model=model)
    CHD(model=model, stress_period_data={0: [[(0, 0), 10.0]]})
    return model, vor


In [ ]:
workspace = project_root / 'examples' / 'mf6' / 'artifacts' / 'pest_first_slice_demo'
if workspace.exists():
    shutil.rmtree(workspace)
workspace.mkdir(parents=True)

model, vor = build_two_cell_pest_forward_model('pest_demo', workspace / 'model')

hk_path = write_gpkg(
    workspace / 'hk.gpkg',
    gpd.GeoDataFrame(
        {'name': ['hk_zone'], 'unit': ['all'], 'k': [5.0]},
        geometry=[Polygon([(0.0, 0.0), (2.0, 0.0), (2.0, 1.0), (0.0, 1.0)])],
        crs=model.vor.crs,
    ),
)

drn_path = write_gpkg(
    workspace / 'drn.gpkg',
    gpd.GeoDataFrame(
        {
            'name': ['east_drn'],
            'group': ['main'],
            'layer': [1],
            'height': [8.5],
            'elev': [8.5],
            'cond': [20.0],
            'min_elev': [0.0],
        },
        geometry=[Polygon([(1.1, 0.1), (1.9, 0.1), (1.9, 0.9), (1.1, 0.9)])],
        crs=model.vor.crs,
    ),
)

drn_builder = mf.DRNFromVector(model=model, vor=vor, shp_gpkg=drn_path, uid='name')
drn_spd = drn_builder.from_vector(
    fields={
        'name': 'name',
        'height_over_btm': 'height',
        'conductance': 'cond',
        'layer': 'layer',
        'min_elev': 'min_elev',
    }
)
Drains(model=model, stress_period_data=drn_spd)

success, _ = model.run_simulation()
print('base model ran:', success)


In [ ]:
points_path = write_gpkg(
    workspace / 'targets.gpkg',
    gpd.GeoDataFrame(
        {'name': ['OBS_A', 'OBS_B'], 'layer': [0, 0], 'weight': [1.0, 1.0]},
        geometry=[Point(0.5, 0.5), Point(1.5, 0.5)],
        crs=model.vor.crs,
    ),
)

targets = mf.HeadTargets(
    locations=points_path,
    values=pd.DataFrame({'per': [0], 'OBS_A': [10.0], 'OBS_B': [8.75]}),
    time_column='per',
)

targets.to_long()


In [ ]:
pest = mf.PestProject(
    model=model,
    name='cumberland_style_forward',
    workspace=workspace / 'template',
    start_datetime='2024-01-01',
)

pest.add_parameter(
    mf.KPilotPointParameter(
        name='hk',
        source=mf.VectorParameterSource(path=hk_path, value_column='k', zone_column='unit'),
        bounds=(0.25, 4.0),
        bounds_mode='multiplier',
        transform='log',
        pp_spacing=5.0,
        geostruct=mf.ExpGeoStruct(range=10.0, transform='log'),
    )
)

pest.add_parameter(
    mf.DrainElevationParameter(
        name='drn_elev',
        source=mf.VectorParameterSource(
            path=drn_path,
            value_column='elev',
            feature_id_column='name',
            group_column='group',
            layer_column='layer',
        ),
        bounds=(-2.0, 2.0),
        bounds_mode='absolute',
    )
)

pest.add_parameter(
    mf.DrainConductanceParameter(
        name='drn_cond',
        source=mf.VectorParameterSource(
            path=drn_path,
            value_column='cond',
            feature_id_column='name',
            group_column='group',
            layer_column='layer',
        ),
        bounds=(0.25, 4.0),
        bounds_mode='multiplier',
        transform='log',
    )
)

pest.add_observation(mf.HeadTargetObservationSpec(targets=targets))
pst = pest.build_pst('first_slice.pst')
print('npar_adj:', pst.npar_adj)
print('nnz_obs:', pst.nnz_obs)


In [ ]:
template = workspace / 'template'
print('template files:')
for path in sorted(template.iterdir()):
    if path.is_file() and path.suffix.lower() in {'.csv', '.json', '.pst', '.py', '.tpl'}:
        print(' -', path.name)

pd.read_json(template / 'pest_forward_config.json')


In [ ]:
initial_heads = pd.read_csv(template / 'hds_simulated_heads.csv')
initial_heads


In [ ]:
hk_frame = pd.read_csv(template / 'hk_pilot_points.csv')
hk_frame['value'] = 2.0
hk_frame.to_csv(template / 'hk_pilot_points.csv', index=False)

drn_elev_frame = pd.read_csv(template / 'drn_elev_drain_elevation.csv')
drn_elev_frame['value'] = 0.75
drn_elev_frame.to_csv(template / 'drn_elev_drain_elevation.csv', index=False)

drn_cond_frame = pd.read_csv(template / 'drn_cond_drain_conductance.csv')
drn_cond_frame['value'] = 0.5
drn_cond_frame.to_csv(template / 'drn_cond_drain_conductance.csv', index=False)

subprocess.run([sys.executable, 'forward_run.py'], cwd=template, check=True)


In [ ]:
rerun_heads = pd.read_csv(template / 'hds_simulated_heads.csv')
rerun_heads


In [ ]:
import flopy

sim = flopy.mf6.MFSimulation.load(sim_ws=str(template), verbosity_level=0)
gwf = sim.get_model(model.name)

print('updated k array:')
print(np.asarray(gwf.npf.k.array, dtype=float))

print('
updated drn data:')
print(gwf.drn.stress_period_data.data[0])


## What This First Slice Covers

This first calibration slice is meant to prove the reusable workflow, not to cover every PEST feature yet.

What is working here:
- `HeadTargets` as a reusable non-PEST-specific target layer
- GIS-defined `K` and `DRN` parameter sources
- template/control-file generation with `PestProject`
- forward-run application of `K` pilot-point multipliers
- forward-run application of drain elevation offsets and conductance multipliers
- regeneration of simulated head target CSVs after each run

Natural next steps after this slice:
- recharge multipliers
- richer observation types
- Cumberland-scale pilot-point layouts and parameter grouping
- prior ensembles / regularization helpers
